JAI SHREE RAM 

In [1]:
%uv pip install -q transformers datasets accelerate evaluate scikit-learn sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [2]:
import transformers, datasets, torch
print(transformers.__version__, datasets.__version__, torch.__version__)
torch.cuda.is_available()

4.56.0 5.0.1 2.8.0+cu129


True

In [3]:
torch.cuda.get_device_name(0)

'Tesla T4'

In [4]:
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
# just a simple pretrained language model

In [ ]:
SEED = 42
MAX_LENGTH = 400
ID2LABEL = {0: "abusive", 1: "non-abusive"}
LABEL2ID = {"abusive": 0, "non-abusive": 1}
OUTPUT_DIR = "/vol/checkpoints/pt_codemix"


In [ ]:
from datasets import load_dataset
from pathlib import Path

# Hinglish code-mix MACD (Bedrock-converted). Keep only text/label to match Davidson.
CODEMIX_DIR = Path("codemix_hinglish")
assert (CODEMIX_DIR / "train.csv").exists(), f"missing {CODEMIX_DIR}/train.csv — sync from S3 first"

def load_codemix(split: str):
    ds = load_dataset("csv", data_files=str(CODEMIX_DIR / f"{split}.csv"))["train"]
    keep = {"text", "label"}
    drop = [c for c in ds.column_names if c not in keep]
    return ds.remove_columns(drop)

macd_train = load_codemix("train")
print(macd_train)
print(macd_train[0])


In [ ]:
macd_val = load_codemix("val")
print(macd_val)


In [ ]:
macd_test = load_codemix("test")
print(macd_test)


In [9]:
macd_train

Dataset({
    features: ['label', 'text'],
    num_rows: 20183
})

In [10]:
davidson = load_dataset("csv",data_files="https://raw.githubusercontent.com/t-davidson/hate-speech-and-offensive-language/master/data/labeled_data.csv")["train"]

Generating train split: 0 examples [00:00, ? examples/s]

In [11]:
def convert_labels(example):
    if example["class"] in [0,1]:
        return {"label": 0}
    else:
        return {"label" : 1}

In [12]:
davidson

Dataset({
    features: ['Unnamed: 0', 'count', 'hate_speech', 'offensive_language', 'neither', 'class', 'tweet'],
    num_rows: 24783
})

In [13]:
davidson = davidson.map(convert_labels)

Map:   0%|          | 0/24783 [00:00<?, ? examples/s]

In [14]:
davidson = davidson.rename_column("tweet","text")

In [15]:
rem = [x for x in davidson.column_names if x not in ["text","label"]]
davidson = davidson.remove_columns(rem)


davidson = davidson.train_test_split(
    test_size=0.10,
    seed=SEED,
)


In [16]:
davidson

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 22304
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2479
    })
})

In [17]:
davidson_train = davidson["train"]
davidson_holdout = davidson["test"]

In [18]:
macd_train.features.type

StructType(struct<label: int64, text: string>)

In [19]:
davidson_train.features.type

StructType(struct<text: string, label: int64>)

In [20]:
from datasets import concatenate_datasets
dataset = concatenate_datasets([davidson_train, macd_train])

In [21]:
dataset=dataset.shuffle(seed=SEED)

In [22]:
print(dataset.column_names)
print(dataset[10])

['text', 'label']
{'text': '🙏जय मां। नैना देवी।🙏 जय मां नैना देवी 🙏जय मां नैना देवी 🙏जय मां नैना देवी 🙏जय मां नैना देवी🙏 जय मां नैना देवी 🙏जय मां नैना देवी🙏 जय मां नैना देवी 🙏जय मां नैना देवी 🙏जय मां नैना देवी 🙏जय मां नैना देवी 🙏जय मां नैना देवी🙏 जय मां नैना देवी🙏🙏🙏🙏🙏🙏🙏', 'label': 1}


In [23]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [24]:
check = tokenizer("यह एक परीक्षण संदेश है", padding=True, truncation=True, return_tensors="pt")


In [25]:
tokenizer.decode(check["input_ids"][0])

'<s> यह एक परीक्षण संदेश है</s>'

In [26]:
def tokenize_function(example):
    return tokenizer(example["text"],  truncation=True)

In [27]:
dataset = dataset.map(tokenize_function,batched=True)

Map:   0%|          | 0/42487 [00:00<?, ? examples/s]

In [28]:
tokenized_train = dataset

In [29]:
tokenized_val = macd_val.map(tokenize_function,batched=True)

Map:   0%|          | 0/6728 [00:00<?, ? examples/s]

In [30]:
tokenized_train

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 42487
})

In [31]:
tokenized_val

Dataset({
    features: ['label', 'text', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 6728
})

In [32]:
tokenized_val[0]

{'label': 0,
 'text': 'Comment box चालू कर झवाडे',
 'input_ids': [0, 16277, 16530, 97491, 1896, 14908, 3097, 18804, 2],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [33]:
features = [
    {
        key : value 
        for key,value in tokenized_train[index].items()
        if key!="text"
    }
    for index in range(4)
]

In [34]:
features


[{'label': 0,
  'input_ids': [0, 5167, 28233, 460, 94618, 2],
  'token_type_ids': [0, 0, 0, 0, 0, 0],
  'attention_mask': [1, 1, 1, 1, 1, 1]},
 {'label': 0,
  'input_ids': [0, 2646, 99511, 4, 935, 4785, 206, 98, 442, 5, 2],
  'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]},
 {'label': 0,
  'input_ids': [0, 92026, 36367, 471, 56980, 471, 36913, 996, 2],
  'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0],
  'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]},
 {'label': 1,
  'input_ids': [0,
   45487,
   10879,
   14131,
   45487,
   45487,
   5195,
   32402,
   10378,
   3277,
   2416,
   6,
   113612,
   113612,
   113612,
   2],
  'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
  'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}]

In [35]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

batch_smoke_test = data_collator(features)
batch_smoke_test

{'input_ids': tensor([[     0,   5167,  28233,    460,  94618,      2,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1],
        [     0,   2646,  99511,      4,    935,   4785,    206,     98,    442,
              5,      2,      1,      1,      1,      1,      1],
        [     0,  92026,  36367,    471,  56980,    471,  36913,    996,      2,
              1,      1,      1,      1,      1,      1,      1],
        [     0,  45487,  10879,  14131,  45487,  45487,   5195,  32402,  10378,
           3277,   2416,      6, 113612, 113612, 113612,      2]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1

In [36]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2,id2label=ID2LABEL,
    label2id=LABEL2ID,
)


config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']


You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


this line actually means that uska jo head hai that is changed to somewhat which is expected by AutoModelForSequenceClassification as per https://huggingface.co/learn/llm-course/chapter2/2


In [37]:
print(model.config.label2id)


{'abusive': 0, 'non-abusive': 1}


classifier's weights and biases are randomly initialized so it expects the model to be retrained on some actual data 

In [38]:
smoke_outputs = model(**batch_smoke_test)

just did a simple forward pass to see the output , this did not include any optimization or back prop


In [39]:
smoke_outputs.loss

tensor(0.7054, grad_fn=<NllLossBackward0>)

In [40]:
smoke_outputs.logits

tensor([[-0.0220,  0.0067],
        [-0.0406,  0.0138],
        [-0.0202, -0.0100],
        [-0.0015, -0.0055]], grad_fn=<AddmmBackward0>)

trainer.train hi actual training karega 

In [41]:
tokenizer.model_max_length

512

In [42]:
model.config.max_position_embeddings

512

In [43]:
from transformers import TrainingArguments


training_args = TrainingArguments("test-trainer", 
                                  eval_strategy="epoch",
                                  save_strategy="epoch",
                                  load_best_model_at_end=True,
                                  metric_for_best_model="f1_macro",
                                  greater_is_better=True,
                                  report_to="none"
                                 )
    

In [44]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

def compute_metrics(eval_predictions):
    logits, labels = eval_predictions
    predictions = np.argmax(logits,axis=-1)

    return {
        "accuracy" : accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels,predictions,average="macro"),
        "precision_macro": precision_score(labels, predictions, average="macro",zero_division=0),
        "recall_macro": recall_score(labels, predictions, average="macro",zero_division=0),
    }


note : compute metrics are not needed for updation of weights because that part is actually done by the loss and the optimizer , the compute metrics is actually for finally saving the model which we feel has done the best over the differnt checkpoints (i mean the epochs in our case)

note that the labels in this is the ground truth value and logits are our 

In [45]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

Detected kernel version 4.19.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [46]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Macro,Recall Macro
1,0.271000,0.415833,0.835018,0.834978,0.837159,0.836216
2,0.224600,0.559014,0.842301,0.841716,0.843837,0.841237
3,0.133700,0.660922,0.844828,0.844682,0.844701,0.844664


TrainOutput(global_step=15933, training_loss=0.22976302214727665, metrics={'train_runtime': 1063.634, 'train_samples_per_second': 119.835, 'train_steps_per_second': 14.98, 'total_flos': 1075359453310800.0, 'train_loss': 0.22976302214727665, 'epoch': 3.0})

internally traning does this:
batch bnao -> forward pass -> calculate loss -> backward pass -> optimizer step -> weights update ->epoch ke end par validation par run -> compute metrics -> checkpoint save

In [47]:
trainer.evaluate()


{'eval_loss': 0.6609223484992981,
 'eval_accuracy': 0.8448275862068966,
 'eval_f1_macro': 0.8446819785126103,
 'eval_precision_macro': 0.8447014929333779,
 'eval_recall_macro': 0.8446638109631843,
 'eval_runtime': 10.4117,
 'eval_samples_per_second': 646.194,
 'eval_steps_per_second': 80.774,
 'epoch': 3.0}

In [48]:
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved to:", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))

Saved to: /vol/checkpoints/pt
['config.json', 'model.safetensors', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']


In [49]:
tokenized_macd_test = macd_test.map(
    tokenize_function,
    batched=True,
)
tokenized_davidson_holdout = davidson_holdout.map(
    tokenize_function,
    batched=True,
)


Map:   0%|          | 0/6728 [00:00<?, ? examples/s]

Map:   0%|          | 0/2479 [00:00<?, ? examples/s]

In [50]:
macd_test_output = trainer.predict(
    tokenized_macd_test
)

print(macd_test_output.metrics)

{'test_loss': 0.6508870124816895, 'test_accuracy': 0.8451248513674198, 'test_f1_macro': 0.8449208838526752, 'test_precision_macro': 0.8448539059613842, 'test_recall_macro': 0.8450028462514444, 'test_runtime': 12.3064, 'test_samples_per_second': 546.709, 'test_steps_per_second': 68.339}


In [51]:
davidson_test_output = trainer.predict(
    tokenized_davidson_holdout
)
print(davidson_test_output.metrics)

{'test_loss': 0.14256446063518524, 'test_accuracy': 0.9677289229528035, 'test_f1_macro': 0.9415530066368812, 'test_precision_macro': 0.9495564505483165, 'test_recall_macro': 0.9340031744560557, 'test_runtime': 2.962, 'test_samples_per_second': 836.93, 'test_steps_per_second': 104.658}


shrink the fine-tuned MiniLM for on device chrome and infernce with ONXXweb runtime !
### Goal :  run inference with ONXX web runtime

### STEPS 
1. measure curr size
2. get ONXX FP32 export with https://huggingface.co/docs/optimum/quicktour#onnx-runtime
3. now quantize this INT8 model
4. measure size and accuracy delta
   

In [52]:
from pathlib import Path

def print_model_size(model_dir, title=None):
    """Print per-file and total on-disk size (MB). Follows symlinks."""
    root = Path(model_dir).resolve()
    if title:
        print(f"=== {title} ===")
    print("resolved:", root)
    assert root.is_dir(), f"missing directory: {root}"

    files = sorted(p for p in root.rglob("*") if p.is_file())
    total = 0
    for p in files:
        mb = p.stat().st_size / 1024**2
        total += mb
        print(f"  {str(p.relative_to(root)):40s} {mb:8.2f} MB")
    print(f"TOTAL: {total:.2f} MB\n")
    return total

print_model_size(OUTPUT_DIR, title="Checkpoint Directory")


=== Checkpoint Directory ===
resolved: /__modal/volumes/vo-g3ni5ic5VpZrHja6I4Qx8s
  config.json                                  0.00 MB
  model.safetensors                          448.84 MB
  special_tokens_map.json                      0.00 MB
  tokenizer.json                              16.29 MB
  tokenizer_config.json                        0.00 MB
  training_args.bin                            0.01 MB
TOTAL: 465.14 MB



465.138952255249

In [53]:
%uv pip install -q "optimum[onnxruntime]" onnx onnxruntime

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [54]:
import optimum
print("optimum ready")

optimum ready


In [ ]:
from pathlib import Path

ONNX_FP32_DIR= "/vol/checkpoints/onnx_fp32_codemix"
Path(ONNX_FP32_DIR).mkdir(parents=True, exist_ok=True)

ONNX_INT8_DIR = "/vol/checkpoints/onnx_int8_codemix"
Path(ONNX_INT8_DIR).mkdir(parents=True, exist_ok=True)


In [56]:
from pathlib import Path
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

# this time we are loading ths model directly 
model_checkpoint = OUTPUT_DIR
save_directory = ONNX_FP32_DIR


# Load a model from transformers and export it to ONNX
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
ort_model = ORTModelForSequenceClassification.from_pretrained(model_checkpoint, export=True)

# Save the ONNX model and tokenizer
ort_model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

Multiple distributions found for package optimum. Picked distribution: optimum-onnx
`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


('/vol/checkpoints/onnx_fp32/tokenizer_config.json',
 '/vol/checkpoints/onnx_fp32/special_tokens_map.json',
 '/vol/checkpoints/onnx_fp32/tokenizer.json')

In [57]:
print_model_size(ONNX_FP32_DIR, "ONNX FP32")

=== ONNX FP32 ===
resolved: /vol/checkpoints/onnx_fp32
  config.json                                  0.00 MB
  model.onnx                                 449.02 MB
  special_tokens_map.json                      0.00 MB
  tokenizer.json                              16.29 MB
  tokenizer_config.json                        0.00 MB
TOTAL: 465.31 MB



465.31155586242676

dynamic int8 quantization :-
> AutoQuantizationConfig + ORTQuantizer -> quanitize(...)

In [58]:
from optimum.onnxruntime.configuration import AutoQuantizationConfig
from optimum.onnxruntime import ORTQuantizer
from pathlib import Path
from transformers import AutoTokenizer


# Define the quantization methodology
qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)
quantizer = ORTQuantizer.from_pretrained(ort_model)

save_directory = ONNX_INT8_DIR


# Apply dynamic quantization on the model
quantizer.quantize(save_dir=save_directory, quantization_config=qconfig)

# same tokenizer 
tokenizer = AutoTokenizer.from_pretrained(ONNX_FP32_DIR)
tokenizer.save_pretrained(ONNX_INT8_DIR)




('/vol/checkpoints/onnx_int8/tokenizer_config.json',
 '/vol/checkpoints/onnx_int8/special_tokens_map.json',
 '/vol/checkpoints/onnx_int8/tokenizer.json')

In [59]:
print_model_size(ONNX_INT8_DIR, "ONNX INT8")

=== ONNX INT8 ===
resolved: /vol/checkpoints/onnx_int8
  config.json                                  0.00 MB
  model_quantized.onnx                       112.79 MB
  ort_config.json                              0.00 MB
  special_tokens_map.json                      0.00 MB
  tokenizer.json                              16.29 MB
  tokenizer_config.json                        0.00 MB
TOTAL: 129.08 MB



129.08484840393066

In [67]:
trainer.model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
trainer.model.to(trainer.args.device)

pt_val_output = trainer.predict(tokenized_val)
print("PyTorch val:", pt_val_output.metrics)

PyTorch val: {'test_loss': 0.6609223484992981, 'test_accuracy': 0.8448275862068966, 'test_f1_macro': 0.8446819785126103, 'test_precision_macro': 0.8447014929333779, 'test_recall_macro': 0.8446638109631843, 'test_runtime': 11.6135, 'test_samples_per_second': 579.326, 'test_steps_per_second': 72.416}


In [71]:
from optimum.onnxruntime import pipeline as ort_pipeline
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(ONNX_INT8_DIR)
int8_model = ORTModelForSequenceClassification.from_pretrained(
    ONNX_INT8_DIR,
    file_name="model_quantized.onnx",
)

classifier = ort_pipeline("text-classification", model=int8_model, tokenizer=tokenizer,device=-1,)

preds = classifier(list(macd_val["text"]), batch_size=64, truncation=True)
pred_ids = [LABEL2ID[p["label"]] for p in preds]
labels = list(macd_val["label"])

int8_acc = accuracy_score(labels, pred_ids)
int8_f1 = f1_score(labels, pred_ids, average="macro")
print("INT8 val:", {"accuracy": int8_acc, "f1_macro": int8_f1})
print("Δ accuracy:", int8_acc - 0.8448275862068966)
print("Δ f1_macro:", int8_f1 - 0.8446819785126103)

Device set to use cpu


INT8 val: {'accuracy': 0.8436385255648038, 'f1_macro': 0.8435110831252078}
Δ accuracy: -0.0011890606420927874
Δ f1_macro: -0.0011708953874024486
